In [1]:
!pip install geemap earthengine-api rioxarray geopandas --quiet

In [2]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import rioxarray as rxr
import pandas as pd 
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS
# ==========================================
# Melacak lokasi folder tempat script ini dijalankan
BASE_DIR = os.getcwd()

# Menentukan lokasi GeoJSON
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # Path di environment Kaggle
    file_geojson = "/kaggle/input/datasets/jerismeteo/kebumen-geojson/33.05_kecamatan.geojson"
else:
    # Path di komputer lokal (asumsi file geojson berada 1 folder dengan script)
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

# Menentukan folder utama untuk output GSMaP
FOLDER_BASE_OUTPUT = os.path.join(BASE_DIR, "data", "gsmap")

tahun_awal = 2025
tahun_akhir = 2026

In [4]:
from kaggle_secrets import UserSecretsClient

# 1. Buka brankas Kaggle dan ambil kunci rahasianya
brankas = UserSecretsClient()
ee_token = brankas.get_secret("EE_TOKEN")

# 2. Buat folder sembunyi di sistem komputer Kaggle
os.makedirs('/root/.config/earthengine', exist_ok=True)

# 3. Tulis ulang kunci tersebut menjadi file kredensial
with open('/root/.config/earthengine/credentials', 'w') as f:
    f.write(ee_token)

In [ ]:
# 1. Inisialisasi Earth Engine
ee.Initialize(project='staklimjerukagung')
# 2. Persiapan Batas Wilayah
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}. Pastikan file ada di folder yang sama dengan script.")

gdf = gpd.read_file(file_geojson)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# ==========================================
# --- KUNCI PERBAIKAN (CLEANING TOPOLOGI) ---
# ==========================================
print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _extract_polygonal(geom):
    if geom is None or geom.is_empty: return None
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys: return None
        return unary_union(polys)
    return None

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty: return None
    geom = geom.buffer(0)
    if geom is None or geom.is_empty: return None
    if not geom.is_valid:
        geom = make_valid(geom)
        geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty or not geom.is_valid: return None
    return geom

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

if gdf.empty:
    raise ValueError("Semua geometri tidak valid setelah proses cleaning. Cek file GeoJSON sumber.")

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])
# ==========================================

# ==========================================
# 3. FUNGSI PERULANGAN UNDUH PER TAHUN (GSMaP) - OPTIMASI PARALEL
# ==========================================
import concurrent.futures

def unduh_satu_jam(tahun, bulan, hari, jam, batas_ee, output_dir):
    # Membentuk objek waktu per jam
    tgl_mulai_obj = datetime(tahun, bulan, hari, jam)
    tgl_akhir_obj = tgl_mulai_obj + timedelta(hours=1)

    # Format string ISO 8601 yang disukai Earth Engine (YYYY-MM-DDTHH:MM:SS)
    tgl_mulai = tgl_mulai_obj.strftime("%Y-%m-%dT%H:%M:%S")
    tgl_akhir = tgl_akhir_obj.strftime("%Y-%m-%dT%H:%M:%S")

    # Penamaan file ditambah atribut jam
    file_name_base = f"gsmap_{hari:02d}_{bulan:02d}_{tahun}_{jam:02d}00"
    tif_path = os.path.join(output_dir, f"{file_name_base}.tif")
    nc_path = os.path.join(output_dir, f"{file_name_base}.nc")

    # Jika file NetCDF sudah ada, lewati
    if os.path.exists(nc_path):
        return f"[{tgl_mulai}] Sudah ada, dilewati..."

    try:
        # 1. Tarik Data GSMaP V8 Operational
        dataset = ee.ImageCollection('JAXA/GPM_L3/GSMaP/v8/operational') \
                    .filter(ee.Filter.date(tgl_mulai, tgl_akhir))

        # Variabel GSMaP adalah 'hourlyPrecipRate'
        precipitation = dataset.select('hourlyPrecipRate').first().clip(batas_ee)

        # 2. Ekspor ke GeoTIFF
        geemap.ee_export_image(
            precipitation,
            filename=tif_path,
            region=batas_ee.geometry(),
            scale=11132,
            file_per_band=False
        )

        # 3. Konversi GeoTIFF ke NetCDF
        with rxr.open_rasterio(tif_path, masked=True) as da_asli:
            da = da_asli.squeeze("band", drop=True) if "band" in da_asli.dims else da_asli
            da.name = "precipitation"

            # Suntikkan metadata waktu
            waktu = pd.to_datetime(tgl_mulai_obj)
            da = da.expand_dims(time=[waktu])

            da.to_netcdf(nc_path)

        # 4. Hapus file TIF
        os.remove(tif_path)
        return f"[{tgl_mulai}] ✓ Berhasil diunduh"

    except Exception as e:
        return f"[{tgl_mulai}] ❌ Error: {e}"


def unduh_gsmap_satu_tahun(tahun, batas_ee, output_base_dir):
    os.makedirs(output_base_dir, exist_ok=True)

    print(f"\n=== Memulai Unduhan GSMaP Tahun {tahun} (Mode Paralel) ===")
    print(f"Output folder utama: {output_base_dir}")

    # Menggunakan ThreadPoolExecutor untuk download paralel
    # max_workers=10 cukup aman untuk tidak memicu rate limit berlebihan dari Earth Engine
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        for bulan in range(1, 13):
            _, jml_hari = calendar.monthrange(tahun, bulan)

            output_dir = os.path.join(output_base_dir, f"{tahun}_{bulan:02d}")
            os.makedirs(output_dir, exist_ok=True)

            print(f"\n--- Mengirim tugas unduhan Bulan {bulan:02d}-{tahun} ({jml_hari} Hari x 24 Jam) ---")
            
            futures = []
            for hari in range(1, jml_hari + 1):
                for jam in range(24):
                    futures.append(
                        executor.submit(unduh_satu_jam, tahun, bulan, hari, jam, batas_ee, output_dir)
                    )
            
            # Menunggu dan mencetak hasil secara asinkron
            for future in concurrent.futures.as_completed(futures):
                print(future.result())


def unduh_gsmap_multi_tahun(tahun_awal, tahun_akhir, batas_ee, output_base_dir):
    if tahun_awal > tahun_akhir:
        raise ValueError("tahun_awal tidak boleh lebih besar dari tahun_akhir")

    os.makedirs(output_base_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"UNDUH GSMaP MULTI-TAHUN: {tahun_awal} - {tahun_akhir}")
    print(f"{'='*60}")

    for tahun in range(tahun_awal, tahun_akhir + 1):
        try:
            folder_tahun = os.path.join(output_base_dir, str(tahun))
            os.makedirs(folder_tahun, exist_ok=True)

            unduh_gsmap_satu_tahun(
                tahun=tahun,
                batas_ee=batas_ee,
                output_base_dir=folder_tahun
            )
        except Exception as e:
            print(f"❌ Gagal memproses tahun {tahun}: {e}")
            continue

    print(f"\n{'='*60}")
    print("SELESAI UNDUH SEMUA TAHUN")
    print(f"{'='*60}")

# ==========================================
# 4. EKSEKUSI FUNGSI
# ==========================================
unduh_gsmap_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    output_base_dir=FOLDER_BASE_OUTPUT
)

Membersihkan geometri GeoJSON yang cacat...
Mengonversi ke Earth Engine...

UNDUH GSMaP MULTI-TAHUN: 2025 - 2026

=== Memulai Unduhan GSMaP Tahun 2025 (Mode Paralel) ===
Output folder utama: d:\Github\Projek_Downscale\data\gsmap\2025

--- Mengirim tugas unduhan Bulan 01-2025 (31 Hari x 24 Jam) ---
[2025-01-01T21:00:00] Sudah ada, dilewati...
[2025-01-01T02:00:00] Sudah ada, dilewati...
[2025-01-01T14:00:00] Sudah ada, dilewati...
[2025-01-01T04:00:00] Sudah ada, dilewati...
[2025-01-02T00:00:00] Sudah ada, dilewati...
[2025-01-01T19:00:00] Sudah ada, dilewati...
[2025-01-01T20:00:00] Sudah ada, dilewati...
[2025-01-01T08:00:00] Sudah ada, dilewati...
[2025-01-02T07:00:00] Sudah ada, dilewati...
[2025-01-01T13:00:00] Sudah ada, dilewati...
[2025-01-02T06:00:00] Sudah ada, dilewati...
[2025-01-01T18:00:00] Sudah ada, dilewati...
[2025-01-01T23:00:00] Sudah ada, dilewati...
[2025-01-01T12:00:00] Sudah ada, dilewati...
[2025-01-01T03:00:00] Sudah ada, dilewati...
[2025-01-02T05:00:00] Suda